## ÍNDICE DEL PROYECTO

* [Introducción](#INTRODUCCIÓN)
* [1. Análisis y Exploración del Dataset](#1.-Análisis-y-Exploración-del-Dataset)
   * [1.1 Carga de Matrices y Limpieza de Estructuras](#1.1.-Carga-de-Matrices-y-Limpieza-de-Estructuras)
   * [1.2 Verificación de Datos Faltantes e Integridad Numérica](#1.2.-Verificación-de-Datos-Faltantes-e-Integridad-Numérica)
   * [1.3 Mapeo de Entidades (CEDIS y Sucursales)](#1.3.-Mapeo-de-Entidades-(CEDIS-y-Sucursales))
   * [1.4 Resumen Estadístico de Distancias y Costos](#1.4.-Resumen-Estadístico-de-Distancias-y-Costos)
* [2. Modelado del Problema Combinatorio (VRP / TSP)](#2-modelado-del-problema-combinatorio-vrp--tsp)
   * [2.1 Formulación Matemática del Problema](#2.1.-Formulación-Matemática-del-Problema)
   * [2.2 Definición de la Función Objetivo de Costo y Distancia](#2.2.-Definición-de-la-Función-Objetivo-de-Costo-y-Distancia)
   * [2.3 Construcción y Evaluación de la Ruta de Línea Base (Baseline)](#2.3.-Construcción-y-Evaluación-de-la-Ruta-de-Línea-Base-(Baseline))
* [3. Desarrollo e Implementación del Algoritmo Heurístico](#3-desarrollo-e-implementación-del-algoritmo-heurístico)
   * [3.1 Justificación Teórica de Recocido Simulado](#3.1.-Justificación-Teórica-de-Recocido-Simulado)
   * [3.2 Operador de Vecindad (Swap / Intercambio 2-Opt)](#3.2.-Operador-de-Vecindad-(Swap-/-Intercambio-2-Opt))
   * [3.3 Función de Validación de Rutas y Restricciones](#3.3.-Función-de-Validación-de-Rutas-y-Restricciones)
   * [3.4 Estructura del Algoritmo de Recocido Simulado (Criterio de Metropolis)](#3.4.-Estructura-del-Algoritmo-de-Recocido-Simulado-(Criterio-de-Metropolis))
* [4. Ejecución de Simulaciones y Registro de Métricas](#4-ejecución-de-simulaciones-y-registro-de-métricas)
   * [4.1 Configuración de Parámetros de Enfriamiento](#4.1.-Configuración-de-Parámetros-de-Enfriamiento)
   * [4.2 Ejecución de la Simulación y Tracking de Costos](#4.2.-Ejecución-de-la-Simulación-y-Tracking-de-Costos)
   * [4.3 Registro de la Mejor Ruta y Comparativa Directa](#4.3.-Registro-de-la-Mejor-Ruta-y-Comparativa-Directa)
* [5. Evaluación de Rendimiento y Comparativa](#5-evaluación-de-rendimiento-y-comparativa)
   * [5.1 Análisis de Convergencia Visual (Curva de Enfriamiento)](#5.1.-Análisis-de-Convergencia-Visual-(Curva-de-Enfriamiento))
   * [5.2 Cálculo de Ahorro Porcentual en Combustible y Distancia](#5.2.-Cálculo-de-Ahorro-Porcentual-en-Combustible-y-Distancia)
* [6. Generación del Mapa Interactivo Geoespacial](#6-generación-del-mapa-interactivo-geoespacial)
   * [6.1 Mapeo de Puntos (CEDIS en Rojo y Sucursales en Azul)](#6.1.-Mapeo-de-Puntos-(CEDIS-en-Rojo-y-Sucursales-en-Azul))
   * [6.2 Trazado y Renderizado de la Polílínea sobre Culiacán](#6.2.-Trazado-y-Renderizado-de-la-Polílínea-sobre-Culiacán)

## Introducción

La distribución logística de insumos de abarrotes en Culiacán, Sinaloa, presenta retos operativos debido a la dispersión geográfica de las sucursales, patrones de tráfico urbano y variaciones en los costos de consumo de combustible. Probar todas las combinaciones de rutas de forma exhaustiva resulta inviable ya que el Problema de Enrutamiento de Vehículos (VRP) es de complejidad $NP$-hard.

En esta libreta se documenta el diseño, modelado e implementación computacional de una solución basada en la metaheurística de **Recocido Simulado (Simulated Annealing)** en Python. El objetivo es minimizar la distancia recorrida y los costos asociados de combustible desde un Centro de Distribución (CEDIS) hacia 90 sucursales de restaurantes, garantizando eficiencia logística, comparabilidad frente a una línea base sin optimizar y visualización geoespacial interactiva.

## 1.- Análisis y Exploración del Dataset

En este primer paso importamos las librerías necesarias y realizamos la lectura de los tres archivos base del proyecto:
- `datos_distribucion_tiendas.xlsx`: Información descriptiva de los 100 nodos (tipo, coordenadas, capacidades).
- `matriz_distancias.xlsx`: Matriz de $100 \times 100$ con las distancias inter-nodo en kilómetros.
- `matriz_costos_combustible.xlsx`: Matriz de $100 \times 100$ con los costos de consumo de combustible en MXN.

### 1.1.- Carga de Matrices y Limpieza de Estructuras

**Funciones y atributos utilizados:**
1. **`pd.read_excel()`**: Carga el contenido de los archivos de Excel en un DataFrame de Pandas.
2. **`.to_numpy()`**: Convierte las matrices bidimensionales de Pandas en arreglos matriciales de NumPy para optimizar sustancialmente la velocidad de cálculo aritmético durante las iteraciones del algoritmo.
3. **`.shape`**: Retorna una tupla con las dimensiones (filas, columnas) de los conjuntos de datos.
4. **`.value_counts()`**: Contabiliza la cantidad de registros por tipo de inmueble (Centro de Distribución vs. Tienda).

In [4]:
import numpy as np
import pandas as pd

# Lectura de los archivos Excel suministrados
df_tiendas = pd.read_excel('./Datasets/datos_distribucion_tiendas.xlsx')
df_distancias = pd.read_excel('./Datasets/matriz_distancias.xlsx')
df_costos = pd.read_excel('./Datasets/matriz_costos_combustible.xlsx')

# Conversión a matrices NumPy para facilitar el procesamiento
matriz_dist = df_distancias.to_numpy()
matriz_costo = df_costos.to_numpy()

# Validación de estructuras de datos y verificación de dimensiones
print('=== VERIFICACIÓN DE ESTRUCTURAS DE DATOS===')
print(f'Dimensiones Matriz Distancias: {matriz_dist.shape} (Esperado: 100x100)')
print(f'Dimensiones Matriz Costos:{matriz_costo.shape} (Esperado: 100x100)')
print('\nDesglose de Entidades en el Dataset:')
print(df_tiendas['Tipo'].value_counts())

# Vista previa de las matrices (Muestra de 5x5 para no saturar la pantalla)
print('\n=== VISTA PREVIA: MATRIZ DE DISTANCIAS (km) ===')
display(df_distancias.iloc[:5, :5])

print('\n=== VISTA PREVIA: MATRIZ DE COSTOS (MXN) ===')
display(df_costos.iloc[:5, :5])

=== VERIFICACIÓN DE ESTRUCTURAS DE DATOS===
Dimensiones Matriz Distancias: (100, 100) (Esperado: 100x100)
Dimensiones Matriz Costos:(100, 100) (Esperado: 100x100)

Desglose de Entidades en el Dataset:
Tipo
Tienda                    90
Centro de Distribución    10
Name: count, dtype: int64

=== VISTA PREVIA: MATRIZ DE DISTANCIAS (km) ===


,Nodo_1,Nodo_2,Nodo_3,Nodo_4,Nodo_5
0,0.000000,22.027423,16.103521,15.988868,12.228125
1,22.027423,0.000000,13.763643,11.832109,19.517618
2,16.103521,13.763643,0.000000,18.401809,6.880980
3,15.988868,11.832109,18.401809,0.000000,20.688967
4,12.228125,19.517618,6.880980,20.688967,0.000000



=== VISTA PREVIA: MATRIZ DE COSTOS (MXN) ===


,Nodo_1,Nodo_2,Nodo_3,Nodo_4,Nodo_5
0,0.000000,3.304114,2.415528,2.398330,1.834219
1,3.304114,0.000000,2.064546,1.774816,2.927643
2,2.415528,2.064546,0.000000,2.760271,1.032147
3,2.398330,1.774816,2.760271,0.000000,3.103345
4,1.834219,2.927643,1.032147,3.103345,0.000000


## 1.2 Verificación de Datos Faltantes e Integridad Numérica

Antes de realizar cualquier cálculo, debemos comprobar que las matrices estén completas y no contengan valores nulos o vacíos (`NaN`).

**Funciones utilizadas:**
1. **`.isnull()`**: Identifica si existen celdas sin datos en los DataFrames.
2. **`.sum().sum()`**: Suma el total de celdas nulas en toda la matriz.

In [5]:
# Contar valores nulos en columnas clave de identificación y ubicación
cols_clave = ['Tipo', 'Nombre', 'Latitud_WGS84', 'Longitud_WGS84']

# Nulos en CEDIS y Tiendas evaluando sus datos de ubicación
nulos_cedis = (
    df_tiendas[df_tiendas['Tipo'] == 'Centro de Distribución'][cols_clave]
    .isnull()
    .sum()
    .sum()
)
nulos_tiendas = (
    df_tiendas[df_tiendas['Tipo'] == 'Tienda'][cols_clave].isnull().sum().sum()
)

# Nulos en matrices de distancia y costo
nulos_dist = df_distancias.isnull().sum().sum()
nulos_costos = df_costos.isnull().sum().sum()

print('=== REVISIÓN DE DATOS FALTANTES ===')
print(f'Valores nulos en CEDIS:        {nulos_cedis}')
print(f'Valores nulos en Tiendas:      {nulos_tiendas}')
print(f'Valores nulos en Distancias:   {nulos_dist}')
print(f'Valores nulos en Costos:       {nulos_costos}')

if nulos_cedis + nulos_tiendas + nulos_dist + nulos_costos == 0:
  print('\nLas matrices están completas y sin datos faltantes.')

=== REVISIÓN DE DATOS FALTANTES ===
Valores nulos en CEDIS:        0
Valores nulos en Tiendas:      0
Valores nulos en Distancias:   0
Valores nulos en Costos:       0

Las matrices están completas y sin datos faltantes.


## 1.3 Mapeo de Entidades (CEDIS y Sucursales)

En este subpaso organizamos las 100 entidades del dataset en dos categorías principales: los puntos de origen o almacenes (**CEDIS**) y las sucursales destino (**Tiendas**).

Para mantener la libreta clara y legible sin saturar la pantalla con los 100 registros, visualizamos únicamente una muestra representativa con los 10 CEDIS disponibles y las primeras 10 tiendas a entregar, mostrando la totalidad de sus columnas.

### Detalles de la Implementación:
1. **Filtro por Tipo de Entidad:** Separamos el DataFrame en `cedis_df` (10 almacenes) y `tiendas_df` (90 sucursales) usando el campo `Tipo`.
2. **Definición de Origen y Destinos:** Se define el **Centro de Distribución 1** (índice 0) como la base de salida y regreso del vehículo, y se extraen los índices de las 90 tiendas a visitar.
3. **Visualización Controlada:** Con la función `.head(10)` se despliegan únicamente los primeros 10 registros de cada grupo manteniendo todas sus variables.

**Funciones utilizadas:**
- **`df['Tipo'] == '...'`**: Filtra el DataFrame según el tipo de instalación.
- **`.head(10)`**: Devuelve las primeras 10 filas de un DataFrame.
- **`display()`**: Renderiza la tabla de Pandas en formato estilizado dentro de Jupyter.
- **`list(range())`**: Genera la secuencia numérica de índices correspondientes a las tiendas.

In [6]:
# # Visualizar la tabla completa de df_tiendas como DataFrame
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#   display(df_tiendas)

# 1. Separar CEDIS y Tiendas
cedis_df = df_tiendas[df_tiendas['Tipo'] == 'Centro de Distribución']
tiendas_df = df_tiendas[df_tiendas['Tipo'] == 'Tienda']

# 2. Selección del CEDIS de origen y lista de sucursales a visitar
cedis_origen_idx = 0
sucursales_indices = list(range(len(cedis_df), len(df_tiendas)))

# 3. Mostrar los 10 Centros de Distribución con todas las columnas
print('=== CENTROS DE DISTRIBUCIÓN (10 CEDIS) ===')
display(cedis_df.head(10))

# 4. Mostrar 10 Sucursales con todas las columnas
print('\n=== MUESTRA DE SUCURSALES A ENTREGAR (10 TIENDAS) ===')
display(tiendas_df.head(10))

print('\n=== RESUMEN DEL MAPEO ===')
print(f'Total de CEDIS en el dataset:    {len(cedis_df)}')
print(f'Total de Tiendas a entregar:     {len(tiendas_df)}')
print(f"CEDIS de Origen Seleccionado:    {df_tiendas.iloc[cedis_origen_idx]['Nombre']}")

=== CENTROS DE DISTRIBUCIÓN (10 CEDIS) ===


,Tipo,Nombre,Latitud_WGS84,Longitud_WGS84,Capacidad_Venta,Capacidad_Almacenamiento,Nivel_Tienda
0,Centro de Distribución,Centro de Distribución 1,24.774908,-107.309857,13648,11639,NaN
1,Centro de Distribución,Centro de Distribución 2,24.846399,-107.380268,19012,26049,NaN
2,Centro de Distribución,Centro de Distribución 3,24.731204,-107.468801,17314,19144,NaN
3,Centro de Distribución,Centro de Distribución 4,24.711617,-107.326765,13358,45741,NaN
4,Centro de Distribución,Centro de Distribución 5,24.820223,-107.358385,12848,48698,NaN
5,Centro de Distribución,Centro de Distribución 6,24.704117,-107.306018,23825,12082,NaN
6,Centro de Distribución,Centro de Distribución 7,24.866489,-107.457532,10976,16140,NaN
7,Centro de Distribución,Centro de Distribución 8,24.736365,-107.463319,17164,25247,NaN
8,Centro de Distribución,Centro de Distribución 9,24.760848,-107.395049,26559,49453,NaN
9,Centro de Distribución,Centro de Distribución 10,24.786389,-107.441754,10869,46781,NaN



=== MUESTRA DE SUCURSALES A ENTREGAR (10 TIENDAS) ===


,Tipo,Nombre,Latitud_WGS84,Longitud_WGS84,Capacidad_Venta,Capacidad_Almacenamiento,Nivel_Tienda
10,Tienda,Tienda 1,24.822371,-107.472101,8257,14731,C
11,Tienda,Tienda 2,24.758429,-107.426728,16490,11928,B
12,Tienda,Tienda 3,24.791214,-107.342965,8611,10359,C
13,Tienda,Tienda 4,24.739935,-107.397153,9557,3106,A
14,Tienda,Tienda 5,24.818483,-107.490710,16438,9924,B
15,Tienda,Tienda 6,24.821509,-107.465895,9552,5547,A
16,Tienda,Tienda 7,24.713010,-107.310223,17509,8514,A
17,Tienda,Tienda 8,24.893126,-107.338321,6519,9224,A
18,Tienda,Tienda 9,24.760923,-107.480466,10881,8635,C
19,Tienda,Tienda 10,24.836847,-107.411970,9333,3711,C



=== RESUMEN DEL MAPEO ===
Total de CEDIS en el dataset:    10
Total de Tiendas a entregar:     90
CEDIS de Origen Seleccionado:    Centro de Distribución 1


### 1.4 Resumen Estadístico de Distancias y Costos

Analizamos los valores mínimos, máximos y promedios de distancia y costo entre todos los puntos para entender la escala geográfica y económica del problema.

**Funciones utilizadas:**
1. **`np.min()`**: Encuentra el valor mínimo (ignorando la diagonal principal de distancia 0).
2. **`np.max()`**: Encuentra la distancia o costo más elevado entre dos nodos.
3. **`np.mean()`**: Calcula el valor promedio.

In [7]:
# Creación de copia de las matrices ignorando los ceros de la diagonal principal (distancia a uno mismo)
dist_sin_ceros = matriz_dist[matriz_dist > 0]
costos_sin_ceros = matriz_costo[matriz_costo > 0]

print('=== RESUMEN ESTADÍSTICO DE DISTANCIAS Y COSTOS ===')
print(f'Distancia Mínima entre nodos: {np.min(dist_sin_ceros):.2f} km')
print(f'Distancia Máxima entre nodos: {np.max(dist_sin_ceros):.2f} km')
print(f'Distancia Promedio:           {np.mean(dist_sin_ceros):.2f} km')
print('-----------------------------------------')
print(f'Costo Mínimo por tramo:       ${np.min(costos_sin_ceros):.2f} MXN')
print(f'Costo Máximo por tramo:       ${np.max(costos_sin_ceros):.2f} MXN')
print(f'Costo Promedio por tramo:     ${np.mean(costos_sin_ceros):.2f} MXN')

=== RESUMEN ESTADÍSTICO DE DISTANCIAS Y COSTOS ===
Distancia Mínima entre nodos: 0.12 km
Distancia Máxima entre nodos: 27.35 km
Distancia Promedio:           11.38 km
-----------------------------------------
Costo Mínimo por tramo:       $0.02 MXN
Costo Máximo por tramo:       $4.10 MXN
Costo Promedio por tramo:     $1.71 MXN


## 2. Modelado del Problema Combinatorio (VRP / TSP)
En esta sección traducimos el problema logístico de Culiacán a un modelo matemático. El objetivo es encontrar la secuencia de visita a las 90 sucursales que minimice el costo de combustible, saliendo y regresando al CEDIS.

### 2.1 Formulación Matemática del Problema

Representamos la red de distribución mediante un grafo completo $G = (V, E)$:
- $V = \{0, 1, 2, \dots, n\}$ es el conjunto de nodos. El nodo $0$ corresponde al CEDIS de origen y los nodos $\{1, \dots, n\}$ representan las 90 sucursales.
- $E = \{(i, j) : i, j \in V, i \neq j\}$ es el conjunto de aristas que conectan cada par de ubicaciones.
- $C_{ij}$ representa el costo de combustible o la distancia en kilómetros para trasladarse del nodo $i$ al nodo $j$.

### 2.2 Definición de la Función Objetivo de Costo y Distancia

**Ecuación del costo:**
El costo total $Z$ se calcula sumando tres partes: el viaje del CEDIS a la primera sucursal, los tramos entre sucursales consecutivas y el regreso de la última sucursal al CEDIS:

$$Z = C_{\text{CEDIS},\pi_1} + \sum_{k=1}^{n-1} C_{\pi_k,\pi_{k+1}} + C_{\pi_n,\text{CEDIS}}$$

**Funciones y operadores utilizados:**
1. `def`: Define la función de evaluación `calcular_costo_ruta`.
2. `for ... in range()`: Recorre la lista de sucursales para evaluar cada tramo consecutivamente.
3. `+=`: Acumula progresivamente el costo de cada tramo en la variable total.

In [8]:
def calcular_costo_ruta(ruta, matriz_evaluacion, origen_idx=0):
  """Calcula el costo o distancia total de recorrer una lista de sucursales,

  partiendo y regresando al CEDIS.
  """
  costo_total = 0.0

  # 1. Del CEDIS a la primera sucursal
  costo_total += matriz_evaluacion[origen_idx][ruta[0]]

  # 2. Tramos entre sucursales
  for i in range(len(ruta) - 1):
    costo_total += matriz_evaluacion[ruta[i]][ruta[i + 1]]

  # 3. De la última sucursal de regreso al CEDIS
  costo_total += matriz_evaluacion[ruta[-1]][origen_idx]

  return costo_total

### 2.3 Construcción y Evaluación de la Ruta de Línea Base (Baseline)

Evaluamos la función objetivo sobre el orden secuencial por defecto de las sucursales (sin aplicar algoritmos de optimización) para establecer la métrica inicial de comparación.

**Funciones utilizadas:**
- `calcular_costo_ruta()`: Evalúa la distancia total en km y el costo en MXN de la ruta inicial.
- `df_tiendas.iloc[]`: Obtiene la información del CEDIS seleccionado.

In [9]:
# CÁLCULO DE LA LÍNEA BASE (RUTA INICIAL) tomando en cuenta el orden secuencial de las sucursales en el dataset
costo_inicial_combustible = calcular_costo_ruta(sucursales_indices, matriz_costo, origen_idx=cedis_origen_idx)
distancia_inicial_km = calcular_costo_ruta(sucursales_indices, matriz_dist, origen_idx=cedis_origen_idx)

print('=== LÍNEA BASE (SIN OPTIMIZAR) ===')
print(f'CEDIS de Origen:               {df_tiendas.iloc[cedis_origen_idx]["Nombre"]}')
print(f'Total de sucursales:           {len(sucursales_indices)}')
print(f'Costo Inicial de Combustible:  ${costo_inicial_combustible:,.2f} MXN')
print(f'Distancia Total Recorrida:     {distancia_inicial_km:,.2f} km')

=== LÍNEA BASE (SIN OPTIMIZAR) ===
CEDIS de Origen:               Centro de Distribución 1
Total de sucursales:           90
Costo Inicial de Combustible:  $149.13 MXN
Distancia Total Recorrida:     994.20 km


## 3. Desarrollo e Implementación del Algoritmo Heurístico
En esta sección construimos los componentes esenciales de la metaheurística de **Recocido Simulado (Simulated Annealing)**, fundamentada en el proceso térmico de cristalización de metales y el Criterio de Aceptación de Metropolis.

### 3.1 Justificación Teórica de Recocido Simulado

El espacio de búsqueda para determinar la secuencia óptima de visita a las sucursales partiendo del CEDIS corresponde a la permutación del conjunto de tiendas:

$$\Omega = n! = 90! \approx 1.48 \times 10^{138} \text{ rutas posibles}$$

#### Desglose de los Elementos de la Fórmula:
- **$\Omega$ (Omega):** Representa el espacio muestral o espacio de soluciones, es decir, el conjunto de todas las rutas posibles.
- **$n$:** Número total de sucursales a entregar ($n = 90$).
- **$!$ (Factorial):** Operador de producto descendente ($90 \times 89 \times 88 \times \dots \times 1$) que calcula la cantidad total de combinaciones u órdenes en los que se pueden organizar los 90 nodos.

Dada la naturaleza $NP$-hard del problema de optimización combinatoria, probar cada permutación mediante búsqueda exhaustiva es computacionalmente inviable. Para resolverlo, empleamos la metaheurística de **Recocido Simulado (Simulated Annealing)**, inspirada en el proceso físico de cristalización y recocido de metales.

#### Fundamentos Teóricos del Algoritmo:
1. **Escape de Óptimos Locales (Efecto Térmico):** A diferencia de los métodos estocásticos de descenso (*Greedy* o Búsqueda Local Estricta) que únicamente aceptan movimientos con $\Delta C < 0$, Recocido Simulado acepta movimientos descendentes y aleatoriamente admite soluciones peores ($\Delta C > 0$). Esto previene que el sistema quede atrapado en mínimos locales al inicio de la optimización.
2. **Evolución Probabilística (De Exploración a Explotación):** En fases de alta temperatura ($T_0$), el algoritmo realiza **exploración global** aceptando variaciones de costo altas. A medida que la temperatura se enfría ($T \to 0$), la probabilidad de aceptar soluciones peores decrece asintóticamente hacia cero, transformando el comportamiento en una **explotación local fina** sobre el mínimo global detectado.
3. **Eficiencia en Memoria y Complejidad O(n):** Mantiene únicamente en memoria tres vectores de permutación (Ruta Actual, Ruta Vecina y Mejor Ruta Global), requiriendo un espacio de almacenamiento $O(n)$ y un tiempo de evaluación por iteración $O(n)$.

### 3.2 Operador de Vecindad (Swap / Intercambio 2-Opt)

Para generar una ruta vecina a partir de la ruta actual, aplicamos un operador estocástico de **intercambio simple (*Swap*)**, seleccionando aleatoriamente dos posiciones de la secuencia e invirtiendo sus lugares.

**Funciones y módulos utilizados:**
1. **`copy()` / `.copy()`**: Duplica la ruta actual para no modificar la solución previa por referencia.
2. **`np.random.choice()`**: Selecciona dos índices distintos de forma aleatoria sin reemplazo.

In [10]:
import random

def generar_vecino_swap(ruta):
  """Genera una nueva ruta intercambiando aleatoriamente dos posiciones. (Operador Swap de Vecindad)."""
  ruta_vecina = ruta.copy()

  # Seleccionar dos posiciones aleatorias distintas para ello creamos dos indices
  idx1, idx2 = random.sample(range(len(ruta)), 2)

  # Intercambiar los elementos en las posiciones seleccionadas
  ruta_vecina[idx1], ruta_vecina[idx2] = ruta_vecina[idx2], ruta_vecina[idx1]

  return ruta_vecina

### 3.3 Función de Validación de Rutas y Restricciones

Garantizamos la validez matemática e higiénica de cada ruta candidata evaluando que contenga exactamente 90 sucursales, que no existan nodos duplicados y que todos sus índices pertenezcan al rango válido.

**Funciones utilizadas:**
1. **`len(set(ruta))`**: Convierte la lista en un conjunto de elementos únicos para identificar duplicados.
2. **`all()`**: Confirma que todos los elementos cumplan la condición de pertenencia.

In [ ]:
def validar_ruta(ruta, total_sucursales=90, min_idx=10, max_idx=99):
  """Verifica la validez de la ruta generada comprobando longitud, unicos y rango de nodos."""
  # 1. Verificar total de nodos en la secuencia
  es_longitud_correcta = len(ruta) == total_sucursales

  # 2. Verificar ausencia de nodos duplicados
  es_sin_duplicados = len(set(ruta)) == total_sucursales

  # 3. Verificar que los índices estén dentro del rango de tiendas (10 a 99)
  en_rango = all(min_idx <= node <= max_idx for node in ruta)

  return es_longitud_correcta and es_sin_duplicados and en_rango

# Prueba de validación sobre la ruta de línea base
print('=== PRUEBA DE VALIDACIÓN DE RUTA ===')
if validar_ruta(sucursales_indices):
  print('La ruta inicial es válida: Contiene 90 sucursales únicas.')

=== PRUEBA DE VALIDACIÓN DE RUTA ===
La ruta inicial es válida: Contiene 90 sucursales únicas.


### 3.4 Estructura del Algoritmo de Recocido Simulado (Criterio de Metropolis)

El motor estocástico del algoritmo se rige por el **Criterio de Aceptación de Metropolis-Hastings**, el cual determina formalmente la transición entre el estado actual $\pi$ y el estado vecino $\pi'$.

#### 1. Cálculo de la Variación de Costo ($\Delta C$):
$$\Delta C = Z(\pi') - Z(\pi)$$

##### Desglose de Elementos:
- **$\Delta C$ (Delta C):** Cambio o diferencia neta en el costo de combustible entre ambas soluciones.
- **$Z(\pi')$:** Evaluación de la función objetivo sobre la ruta vecina candidata $\pi'$.
- **$Z(\pi)$:** Evaluación de la función objetivo sobre la ruta actual $\pi$.
- **Criterio:**
  - Si $\Delta C \le 0$: La nueva ruta es de igual o menor costo. Se **acepta incondicionalmente** ($\pi \leftarrow \pi'$).
  - Si $\Delta C > 0$: La nueva ruta incrementa el costo. Se evalúa su aceptación probabilística.

---

#### 2. Ecuación de Probabilidad de Metropolis ($P$):
$$P(\Delta C, T) = \exp\left(-\frac{\Delta C}{T}\right) = e^{-\frac{\Delta C}{T}}$$

##### Desglose de Elementos:
- **$P(\Delta C, T)$:** Probabilidad calculada (un valor continuo entre $0$ y $1$) de aceptar la ruta más costosa.
- **$e$ o $\exp(\cdot)$:** Función exponencial en base e (número de Euler $\approx 2.71828$).
- **$-$ (Signo Negativo):** Invierte el cociente para garantizar que a mayor sobrecosto ($\Delta C$), menor sea la probabilidad de aceptación.
- **$\Delta C$:** Sobrecosto positivo generado por la ruta candidata ($Z(\pi') - Z(\pi) > 0$).
- **$T$:** Temperatura actual del sistema ($T > 0$).
- **Prueba de Hipótesis Estocástica:**
  Se genera un número aleatorio uniforme $r \sim U(0, 1)$. Si $r < P(\Delta C, T)$, el movimiento con sobrecosto es **aceptado** para continuar explorando.

---

#### 3. Esquema de Enfriamiento Geométrico:
$$T_{k+1} = \alpha \cdot T_k \quad \text{con } 0.80 \le \alpha < 1.0$$

##### Desglose de Elementos:
- **$T_{k+1}$:** Temperatura que tendrá el sistema en la siguiente etapa o bloque de iteraciones.
- **$T_k$:** Temperatura actual en la etapa $k$.
- **$\alpha$ (Alfa):** Coeficiente constante de desaceleración térmica o tasa de enfriamiento (típicamente entre $0.80$ y $0.995$).
- **$\cdot$ (Multiplicación):** Aplica una reducción porcentual constante en cada paso (por ejemplo, con $\alpha = 0.995$, la temperatura se reduce un $0.5\%$ en cada iteración).

A medida que $T \to 0$, el exponente $-\frac{\Delta C}{T} \to -\infty$, haciendo que $P \to 0$. Esto obliga al algoritmo a volverse estricto y convergente hacia la mejor solución encontrada.

**Funciones y operadores utilizados:**
- **`math.exp()`**: Evalúa la función exponencial $e^{-x}$ para el Criterio de Metropolis.
- **`random.random()`**: Muestra la variable aleatoria $r \sim U(0, 1)$ para la prueba de hipótesis estocástica.

In [14]:
import math
import random

def ejecutar_recocido_simulado(
    ruta_inicial,
    matriz_eval,
    origen_idx=0,
    temp_inicial=1000.0,
    temp_minima=1e-3,
    alfa=0.995,
    iter_por_temp=100,
):
  """Ejecuta la metaheurística de Recocido Simulado para optimizar la secuencia de entrega."""
  # 1. Estado inicial y evaluación de partida
  # Creamos una copia explícita de la ruta inicial para no modificar la lista original en memoria
  ruta_actual = ruta_inicial.copy()

  # Calculamos la distancia o costo inicial consumido por la secuencia base sin optimizar
  costo_actual = calcular_costo_ruta(
      ruta_actual, matriz_eval, origen_idx=origen_idx
  )

  # 2. Registro de la mejor solución global
  # Inicializamos las variables que guardarán la mejor ruta y costo encontrados a lo largo de la simulación
  mejor_ruta = ruta_actual.copy()
  mejor_costo = costo_actual

  # 3. Control térmico y seguimiento de convergencia
  # Establecemos la temperatura abstracta inicial (T) y creamos el historial para la curva de aprendizaje
  temperatura = temp_inicial
  historial_costos = [costo_actual]

  # 4. Bucle principal de enfriamiento (Criterio de parada por T_minima)
  # El ciclo continúa reduciendo la temperatura hasta que T sea menor o igual al umbral mínimo (1e-3)
  while temperatura > temp_minima:

    # Bucle secundario: Ejecuta un bloque fijado de iteraciones exploratorias a la misma temperatura T
    for _ in range(iter_por_temp):

      # Generamos una solución candidata (vecina) intercambiando dos sucursales al azar (operador Swap)
      ruta_vecina = generar_vecino_swap(ruta_actual)

      # Evaluamos el costo total en MXN o km de la ruta vecina en la matriz
      costo_vecino = calcular_costo_ruta(
          ruta_vecina, matriz_eval, origen_idx=origen_idx
      )

      # Calculamos la diferencia neta de costo (Delta_C = Costo_Nuevo - Costo_Actual)
      delta_costo = costo_vecino - costo_actual

      # Criterio de aceptación de Metropolis-Hastings
      # Caso A (delta_costo < 0): La nueva ruta reduce el costo (mejora). Se acepta automáticamente.
      # Caso B (delta_costo > 0): La nueva ruta incrementa el costo (empeora). Se acepta solo si un número
      # aleatorio uniforme r ~ U(0,1) es menor que la probabilidad exponencial P = exp(-Delta_C / T).
      if delta_costo < 0 or random.random() < math.exp(
          -delta_costo / temperatura
      ):
        ruta_actual = ruta_vecina.copy()
        costo_actual = costo_vecino

        # Si la nueva solución aceptada supera el récord global previo, actualizamos la mejor marca
        if costo_actual < mejor_costo:
          mejor_costo = costo_actual
          mejor_ruta = ruta_actual.copy()

      # Almacenamos el costo del estado actual para posteriormente trazar la gráfica de enfriamiento
      historial_costos.append(costo_actual)

    # Esquema de reducción térmica geométrica
    # Disminuimos progresivamente la temperatura aplicando el factor alfa (T_k+1 = alfa * T_k)
    temperatura *= alfa

  # Retornamos la secuencia optimizada, su costo mínimo asociado y la trayectoria de costos
  return mejor_ruta, mejor_costo, historial_costos